# Parameter estimation and fitting (Part 2)
This is a Python notebook in which you will practice the concepts learned during the lectures.

## Startup ROOT
Import the ROOT module: this will activate the integration layer with the notebook automatically

In [1]:
import ROOT
import numpy as np
print(ROOT.gROOT.GetVersion())

6.38.00


## Use user-defined python function as TF1 for fitting
Functions defined as a _usual_ python function, can be used as well to fit any histogram or graph. However, those functions need a specific signatures. Variables and parameter argumenst MUST be arrays (numpy, array or list)

Inject into the interpreter the functions.

In [2]:
# Define functions for fitting 
#  Quadratic background function
def background(x,par):
    # Note that x and par MUST be arrays 
    return par[0] + par[1]*x[0] + par[2]*x[0]*x[0]

# Lorenzian Peak function
def lorentzianPeak(x, par):    
    # Note that x and par MUST be arrays     
    return (0.5*par[0]*par[1]/ROOT.TMath.Pi()) /  \
        ROOT.TMath.Max(1.e-10,(x[0]-par[2])*(x[0]-par[2])+ 0.25*par[1]*par[1])
    
#  Sum of background and peak function
def fitFunction(x,par):
    # Note that x and par MUST be arrays
    # Note the direct slicing will fail: 
    # https://root-forum.cern.ch/t/how-do-i-handle-double-t-buffer-objects/5482/3
    # IMPORTANT: par shuould be an array of 6 elements: 
    #   * par[0:2]: parameters for the background function
    #   * par[3:5]: parameters for the lorentzian function
    return background(x, par)+lorentzianPeak(x, [par[3],par[4],par[5]])


Construct the histogram containing the input data

In [3]:
nbins = 60
data = [ 6,1,10,12,6,13,23,22,15,21,
         23,26,36,25,27,35,40,44,66,81,
         75,57,48,45,46,41,35,36,53,32,
         40,37,38,31,36,44,42,37,32,32,
         43,44,35,33,33,39,29,41,32,44,
         26,39,29,35,32,21,21,15,25,15 ]
xlow = 0
xup = 3

# build the histogram to be filled with the data
histo_name='histo'
histo_title='Lorentzian Peak on Quadratic Background'

# Fill the content (SetBinContent) of the data list into the histogram
h1 = ROOT.TH1F(histo_name, histo_title, nbins, xlow, xup)
for i in range(len(data)):
    h1.SetBinContent(i+1, data[i])  # bin numbering starts at 1
    h1.SetBinError(i+1, ROOT.TMath.Sqrt(data[i]))  # Poisson errors
h1.GetXaxis().SetTitle("x")
h1.GetYaxis().SetTitle("Counts")
# Draw the histogram
c1 = ROOT.TCanvas("c1", "Histogram", 800, 600)
h1.Draw("E1P")
c1.Draw()

Create the function and try to fit it without setting any parameter

In [4]:
nparams = 6
fitFcn = ROOT.TF1('fitFcn', fitFunction, xlow, xup, nparams)

# Fit the function
h1.Fit(fitFcn)

# Draw it
c2 = ROOT.TCanvas("c2", "Fit Function", 800, 600)
h1.Draw("E1P")
fitFcn.Draw("SAME")
c2.Draw()

****************************************
         Invalid FitResult  (status = 2 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       140.05
NDf                       =           54
Edm                       =  6.74617e-18
NCalls                    =           84
p0                        =    -0.860512   +/-   0           
p1                        =      54.2112   +/-   0           
p2                        =     -16.4868   +/-   0           
p3                        =            0   +/-   0           
p4                        =            0   +/-   0           
p5                        =            0   +/-   0           


Warning in <Fit>: Abnormal termination of minimization.


Less than optimal. Set parameters and fit again, draw histogram with error bars

In [5]:
# Set the parameter 4 (Lorentzian widht)to 0.2
fitFcn.SetParameter(4, 0.2)

# Set the parameter 5 (Lorentzian peak) to 1
fitFcn.SetParameter(5, 1)

# Fit it again
h1.Fit(fitFcn)

# And draw it with errors
c2 = ROOT.TCanvas("c2", "Fit Function", 800, 600)
h1.Draw("E1P")
c2.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      58.9284
NDf                       =           54
Edm                       =  3.83605e-07
NCalls                    =          247
p0                        =    -0.864958   +/-   0.89178     
p1                        =       45.843   +/-   2.6419      
p2                        =     -13.3213   +/-   0.976839    
p3                        =      13.8086   +/-   2.17672     
p4                        =     0.172326   +/-   0.0358153   
p5                        =     0.987281   +/-   0.0112683   


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2


Much better. Now time to beautify the plot. Construct a TF1 for the background and Lorentzian functions and draw them in the same canvas.
We save the fit results and set the parameters of the functions accordingly

In [6]:
# Get the parameters from fitFcn
pars = []
for i in range(nparams):
    pars.append(fitFcn.GetParameter(i))

# Create a TF1 with only the background component (3 parameters)
backFcn = ROOT.TF1('backFcn', background, xlow, xup, 3)

# And set the relevant parameters already fitted
backFcn.SetParameters(pars[0], pars[1], pars[2])

# Set the line color to green (or something you like)
backFcn.SetLineColor(ROOT.kGreen+2)

# And draw it in the same canvas
backFcn.Draw('Same')
c2.Draw()

In [7]:
# Create a TF1 with only the signal component (3 parameters)
signalFcn = ROOT.TF1('signalFcn', lorentzianPeak, xlow, xup, 3)
# Change the color of the line
signalFcn.SetLineColor(ROOT.kRed+2)

# Set the parameters (par variable)
signalFcn.SetParameters(pars[3], pars[4], pars[5])

# And draw it in the same canvas
signalFcn.Draw('Same')
c2.Draw()

We can now add a legend

In [8]:
legend = ROOT.TLegend(0.45, 0.65, 0.73, 0.85)
# Set legend font to 72 and the text size to 0.04
legend.SetTextFont(72)
legend.SetTextSize(0.04)

# Add the different entries: histo is 'Data'; backFcn is 'Background Fit', 
# signalFcn is 'Signal Fit' and fitFcn is 'Global Fit'. All of them should show
# the Line and in the data case the error bars as well
legend.AddEntry(h1, 'Data', 'lep')
legend.AddEntry(backFcn, 'Background Fit', 'l')
legend.AddEntry(signalFcn, 'Signal Fit', 'l')
legend.AddEntry(fitFcn, 'Global Fit', 'l')


# Draw the legend
legend.Draw('Same')
c2.Draw()